# 03 — Phase 3 Ops verification

Implement and verify each op one at a time. Each test cell compares your nanograd output against PyTorch — they should match to ~6 decimal places.

Order:
1. `exp` (in `engine.py`)
2. `log` (in `engine.py`)
3. `softmax` (in `ops.py`)
4. `cross_entropy` (in `ops.py`)

Implement → run the corresponding test cell → if it passes, move on.

In [ ]:
import sys
sys.path.insert(0, '..')

import importlib
import nanograd.engine
import nanograd.ops
importlib.reload(nanograd.engine)  # re-load after edits without restarting kernel
importlib.reload(nanograd.ops)

from nanograd.engine import Value
from nanograd.ops import softmax, cross_entropy
import torch
import math

print('imports ok')

## Test 1 — `exp`

After implementing `exp` in `engine.py`, run this cell.  
Expected: forward and gradient match PyTorch.

In [ ]:
# nanograd
x_n = Value(2.0)
y_n = x_n.exp()
y_n.backward()

# pytorch
x_t = torch.tensor(2.0, requires_grad=True)
y_t = torch.exp(x_t)
y_t.backward()

print(f'forward:  nanograd={y_n.data:.6f}  torch={y_t.item():.6f}')
print(f'gradient: nanograd={x_n.grad:.6f}  torch={x_t.grad.item():.6f}')

assert abs(y_n.data - y_t.item()) < 1e-6
assert abs(x_n.grad - x_t.grad.item()) < 1e-6
print('exp PASS')

## Test 2 — `log`

After implementing `log` in `engine.py`.

In [ ]:
x_n = Value(3.0)
y_n = x_n.log()
y_n.backward()

x_t = torch.tensor(3.0, requires_grad=True)
y_t = torch.log(x_t)
y_t.backward()

print(f'forward:  nanograd={y_n.data:.6f}  torch={y_t.item():.6f}')
print(f'gradient: nanograd={x_n.grad:.6f}  torch={x_t.grad.item():.6f}')

assert abs(y_n.data - y_t.item()) < 1e-6
assert abs(x_n.grad - x_t.grad.item()) < 1e-6
print('log PASS')

## Test 3 — `softmax` forward

After implementing `softmax` in `ops.py`. First just check the forward values.

In [ ]:
logits = [Value(-1.2), Value(0.8), Value(3.5)]
probs = softmax(logits)

logits_t = torch.tensor([-1.2, 0.8, 3.5])
probs_t = torch.softmax(logits_t, dim=0)

for i, (p, pt) in enumerate(zip(probs, probs_t)):
    print(f'p[{i}]:  nanograd={p.data:.6f}  torch={pt.item():.6f}')
    assert abs(p.data - pt.item()) < 1e-6

total = sum(p.data for p in probs)
print(f'sum probs = {total:.6f}  (should be 1.0)')
assert abs(total - 1.0) < 1e-6
print('softmax forward PASS')

## Test 4 — `softmax` gradient

Backward through softmax — sum the probabilities and call `.backward()`. Gradients should match PyTorch.

In [ ]:
# nanograd
logits = [Value(-1.2), Value(0.8), Value(3.5)]
probs = softmax(logits)
# scalar to backward from: pretend we care about the log-prob of class 1
loss = -(probs[1].log())
loss.backward()

# pytorch
logits_t = torch.tensor([-1.2, 0.8, 3.5], requires_grad=True)
probs_t = torch.softmax(logits_t, dim=0)
loss_t = -torch.log(probs_t[1])
loss_t.backward()

print(f'loss: nanograd={loss.data:.6f}  torch={loss_t.item():.6f}')
for i, (l, lt) in enumerate(zip(logits, logits_t)):
    print(f'logit[{i}].grad:  nanograd={l.grad:.6f}  torch={lt.grad.item():.6f}')
    assert abs(l.grad - lt.grad.item()) < 1e-5
print('softmax gradient PASS')

## Test 5 — `cross_entropy`

After implementing `cross_entropy` in `ops.py`. The whole pipeline forward + backward, compared to PyTorch's `F.cross_entropy`.

In [ ]:
# nanograd
logits = [Value(1.0), Value(2.0), Value(3.0)]
loss = cross_entropy(logits, target_idx=1)
loss.backward()

# pytorch
logits_t = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
loss_t = torch.nn.functional.cross_entropy(logits_t.unsqueeze(0), torch.tensor([1]))
loss_t.backward()

print(f'loss: nanograd={loss.data:.6f}  torch={loss_t.item():.6f}')
for i, (l, lt) in enumerate(zip(logits, logits_t)):
    print(f'logit[{i}].grad:  nanograd={l.grad:.6f}  torch={lt.grad.item():.6f}')

assert abs(loss.data - loss_t.item()) < 1e-6
for l, lt in zip(logits, logits_t):
    assert abs(l.grad - lt.grad.item()) < 1e-5
print('\ncross_entropy PASS')
print('\nPHASE 3 COMPLETE')